In [0]:
# Questo notebook crea la tabella Gold finale del progetto Meteo ed Energia.
#
# Legge la parte meteo aggregata dalla tabella:
# progetto_meteo.gold.dati_aggregati
#
# Legge la parte energia aggregata dalla tabella:
# progetto_meteo.silver.energy_block
#
# Poi unisce i due blocchi usando le chiavi comuni:
# - Area
# - Data
# - Ora
#
# Il risultato viene salvato nella tabella finale:
# progetto_meteo.gold.dati_studio
#
# Questa tabella è il dataset principale per analisi e visualizzazioni.
# Contiene, per ogni macroarea, giorno e ora:
# - dati meteorologici aggregati;
# - produzione solare;
# - produzione idrica;
# - produzione eolica.
#
# La tabella finale permette di studiare il rapporto tra condizioni meteo
# e produzione rinnovabile in Italia.
#
# Nota importante:
# il join è un left join sulla parte meteo.
# Questo significa che la tabella finale segue la copertura di gold.dati_aggregati.
# Se per una riga meteo non viene trovata energia corrispondente,
# Solare, Idrico ed Eolico vengono riempiti a 0.
#
# Questi zeri rendono stabile la tabella per analisi e grafici,
# ma vanno interpretati con attenzione:
# uno zero può indicare produzione nulla oppure assenza di corrispondenza nel join.

from pyspark.sql import functions as F


# Imposto catalogo e tabelle del progetto.
catalogo = "progetto_meteo"

tabella_dati_aggregati = f"{catalogo}.gold.dati_aggregati"
tabella_energy_block = f"{catalogo}.silver.energy_block"
tabella_dati_studio = f"{catalogo}.gold.dati_studio"


# Seleziono il catalogo del progetto.
spark.sql(f"USE CATALOG {catalogo}")


# Leggo i dati meteo aggregati.
# Questa tabella contiene temperatura, umidità, vento e precipitazioni
# già aggregati per Area, Data e Ora.
df_meteo = spark.table(tabella_dati_aggregati)


# Leggo i dati energia aggregati.
# Questa tabella contiene Solare, Idrico ed Eolico
# già aggregati per Area, Data e Ora.
df_energia = spark.table(tabella_energy_block)


# Controllo che le tabelle di partenza abbiano dati.
# Se una delle due è vuota, interrompo il notebook con un messaggio chiaro.
righe_meteo = df_meteo.count()
righe_energia = df_energia.count()

if righe_meteo == 0:
    raise Exception(f"Nessun dato trovato in {tabella_dati_aggregati}. Esegui prima Gold_Dati_Aggregati.")

if righe_energia == 0:
    raise Exception(f"Nessun dato trovato in {tabella_energy_block}. Esegui prima Patcher_Energy_Block.")


# Unisco meteo ed energia su Area, Data e Ora.
# Uso left join perché la Gold finale deve seguire la copertura meteo.
df_dati_studio = (
    df_meteo.alias("m")
    .join(
        df_energia.alias("e"),
        on=["Area", "Data", "Ora"],
        how="left"
    )
    .select(
        F.col("Area"),
        F.col("Data"),
        F.col("Ora"),
        F.col("Temperatura"),
        F.col("Umidita"),
        F.col("Velocita_Vento"),
        F.col("Precipitazioni"),
        F.col("Solare"),
        F.col("Idrico"),
        F.col("Eolico")
    )
    .fillna({
        "Solare": 0.0,
        "Idrico": 0.0,
        "Eolico": 0.0
    })
)


# Controllo che il join abbia prodotto righe.
# Se il risultato è vuoto, non aggiorno la tabella finale.
righe_studio = df_dati_studio.count()

if righe_studio == 0:
    raise Exception("Join completato, ma nessuna riga prodotta. Non aggiorno dati_studio.")


# Salvo il risultato nella tabella Gold finale.
# Uso overwrite perché dati_studio deve essere sempre coerente
# con meteo aggregato ed energy_block.
df_dati_studio.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(
    tabella_dati_studio
)


# Controllo finale.
# Mostro un campione ordinato della Gold finale.
display(
    spark.table(tabella_dati_studio)
    .orderBy("Area", "Data", "Ora")
    .limit(100)
)


# Stampo un riepilogo finale della creazione della tabella dati_studio.
print(f"Dati studio completati. Tabella aggiornata: {tabella_dati_studio}")
print(f"Righe meteo aggregate lette: {righe_meteo}")
print(f"Righe energy_block lette: {righe_energia}")
print(f"Righe salvate in dati_studio: {spark.table(tabella_dati_studio).count()}")